# Phi-3 Mini 3.82B — Edge Benchmark Analysis
**Model:** Phi-3 Mini 3.82B Instruct Q4_K_M  
**Backend:** CPU-only (llama.cpp, 12 threads)  
**Rationale for selection:** Achieved a perplexity of 6.82 in Phase 1 — nearly half that of Llama 3.2 3B (12.33) — indicating substantially better output quality and language understanding.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

plt.rcParams.update({
    'figure.facecolor': '#0d0f14', 'axes.facecolor': '#13161e',
    'axes.edgecolor': '#252a38', 'axes.labelcolor': '#e2e8f0',
    'xtick.color': '#64748b', 'ytick.color': '#64748b',
    'text.color': '#e2e8f0', 'grid.color': '#252a38',
    'grid.linewidth': 0.8, 'font.family': 'monospace',
    'axes.titlesize': 12, 'axes.labelsize': 10,
})

ACCENT  = '#f472b6'  # Pink accent for Phi-3
PLOTS   = Path('../results/plots')
PLOTS.mkdir(parents=True, exist_ok=True)

perf = pd.read_csv('../results/raw/performance_phase1.csv')
ppl  = pd.read_csv('../results/raw/perplexity_phase1.csv')

display(perf)
display(ppl)

## 1. Throughput

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
fig.suptitle('Phi-3 Mini 3.82B Q4_K_M — Throughput (CPU, 12 threads)', fontweight='bold')

metrics = [('pp512_mean', 'pp512_std', 'Prompt Processing (pp512)', 't/s', 130),
           ('tg128_mean', 'tg128_std', 'Token Generation (tg128)', 't/s', 25)]

for ax, (mean_col, std_col, title, ylabel, ylim) in zip(axes, metrics):
    val, err = perf[mean_col].values[0], perf[std_col].values[0]
    ax.bar(['Phi-3 Mini'], [val], yerr=[err], capsize=8,
           color=ACCENT, edgecolor='none', width=0.35,
           error_kw=dict(ecolor='#64748b', lw=1.5))
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel(ylabel)
    ax.set_ylim(0, ylim)
    ax.grid(axis='y', alpha=0.4)
    ax.text(0, val + err + ylim*0.02, f'{val:.2f}', ha='center',
            fontsize=12, fontweight='bold', color='#e2e8f0')

plt.tight_layout()
plt.savefig(PLOTS / 'phi3_throughput.png', dpi=150, bbox_inches='tight', facecolor='#0d0f14')
plt.show()

## 2. Memory Breakdown

In [ ]:
# From llama.cpp memory breakdown logs
labels  = ['Model\nweights', 'KV cache', 'Compute\nbuffer']
values  = [2281, 768, 88]   # MiB
colors  = ['#3b4f78', ACCENT, '#475569']

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))

axes[0].bar(labels, values, color=colors, edgecolor='none', width=0.5)
axes[0].set_ylabel('MiB')
axes[0].set_title('Memory by Component', fontweight='bold')
axes[0].grid(axis='y', alpha=0.4)
for i, v in enumerate(values):
    axes[0].text(i, v+10, f'{v} MiB', ha='center', fontsize=9, fontweight='bold')

wedge_props = dict(edgecolor='#0d0f14', linewidth=2)
axes[1].pie(values, labels=labels, colors=colors, autopct='%1.0f%%',
            wedgeprops=wedge_props, textprops={'color': '#e2e8f0', 'fontsize': 9})
axes[1].set_title(f'Peak RAM: {perf["peak_rss_mib"].values[0]:.0f} MiB total', fontweight='bold')

plt.tight_layout()
plt.savefig(PLOTS / 'phi3_memory.png', dpi=150, bbox_inches='tight', facecolor='#0d0f14')
plt.show()

print('Note: KV cache accounts for ~30% of total RAM at ctx-size 512.')
print('This is because Phi-3 Mini uses full multi-head attention (32 heads), not GQA.')

## 3. Perplexity — Per-Chunk Profile

In [ ]:
chunks = [5.4432,5.9309,6.7324,7.1845,7.3546,7.2275,7.2001,7.1537,7.5245,7.5756,
          7.6241,7.5332,7.2713,7.2728,7.4192,7.1824,7.0784,7.1156,6.8317,6.8915,6.8152]

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(range(1, len(chunks)+1), chunks, 'o-', color=ACCENT, linewidth=2, markersize=5)
ax.fill_between(range(1, len(chunks)+1), chunks, alpha=0.1, color=ACCENT)
ax.axhline(y=ppl['ppl'].values[0], color=ACCENT, linestyle='--', alpha=0.6, linewidth=1.2,
           label=f'Final PPL = {ppl["ppl"].values[0]}')
ax.set_xlabel('Chunk index')
ax.set_ylabel('Perplexity')
ax.set_title('Per-Chunk Perplexity — Phi-3 Mini (Wikitext-2, ctx=512)', fontweight='bold')
ax.legend(fontsize=9, framealpha=0.3)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(PLOTS / 'phi3_perplexity.png', dpi=150, bbox_inches='tight', facecolor='#0d0f14')
plt.show()

print(f'Final PPL: {ppl["ppl"].values[0]} ± {ppl["ppl_uncertainty"].values[0]}')
print('The curve stabilises quickly after chunk 4, indicating consistent generalisation.')

## 4. Interpretation

### Why Phi-3 Mini was chosen
Despite being slower and using more memory than Llama 3.2 3B, Phi-3 Mini achieves a **perplexity of 6.82** — nearly half that of Llama 3.2 (12.33). This represents a substantial quality advantage that is highly likely to translate to better reasoning, instruction following, and generation coherence on downstream tasks.

| Metric | Value | vs Llama 3.2 3B |
|--------|-------|------------------|
| pp512 | 72.66 t/s | −34% |
| tg128 | 15.95 t/s | −17% |
| Peak RAM | 2,578 MiB | +22% |
| Perplexity (PPL) | **6.82** | **−45% (better)** |

### Architecture notes
Phi-3 Mini uses standard multi-head attention with 32 KV heads, which is the primary cause of its larger KV cache (768 MiB at ctx=512). Microsoft trained it on a carefully curated high-quality dataset, which is likely responsible for its strong language modelling quality relative to parameter count.

### Perplexity stability
The per-chunk PPL curve rises briefly in chunks 1–4 (context build-up) then stabilises around 7.0–7.6 for the bulk of the corpus. This flat profile indicates consistent generalisation across diverse Wikipedia content.

### Next steps
1. Run GSM8K (8-shot) — Phi-3 Mini was specifically trained for reasoning and is expected to perform well
2. Run HumanEval — Microsoft reported strong coding ability in the Phi-3 technical report
3. Test at ctx-size 2048 and 4096 (Phi-3 Mini trained at 4096)
4. Compare Q4_K_M vs Q5_K_M to assess the quality/size trade-off